# 🚀 Đào tạo mạng SOTA CTR-GCN cho Hệ Thống Báo Động (Transfer Learning)

Notebook này được thiết kế để chạy tự động quy trình chuẩn hóa. Tất cả code logic đã được tách ra thành file `.py` chuyên nghiệp.

In [ ]:
# 1. KẾT NỐI VỚI GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. TẢI MÃ NGUỒN TỪ GITHUB (Tự động đồng bộ bản mới nhất)
!if [ -d "realtime-abnormal-behavior-detection" ]; then cd realtime-abnormal-behavior-detection && git fetch origin && git reset --hard origin/dev; else git clone -b dev https://github.com/NBasLongz/realtime-abnormal-behavior-detection.git; fi
%cd /content/realtime-abnormal-behavior-detection
!pip install -r requirements.txt
!pip install onnxruntime

In [ ]:
# 3. CHÉP DỮ LIỆU TỪ DRIVE VÀO PROJECT
!cp -r /content/drive/MyDrive/Abnormal_Data/raw/* data/raw/

In [ ]:
# 4. TRÍCH XUẤT KHUNG XƯƠNG (Pose Estimation)
!python scripts/data_processing/build_sequences.py

In [ ]:
# 5. CHUYỂN ĐỔI DATA SANG CHUẨN GCN [N, C, T, V] VÀ AUGMENTATION (x4 Dữ Liệu)
!python scripts/data_processing/export_gcn_dataset.py

In [ ]:
# 6. TẢI WEIGHTS CỦA MÔ HÌNH NTU-RGB+D ĐỂ TRANSFER LEARNING
!mkdir -p weights/classification
!wget -O weights/classification/ctrgcn_ntu_120.pt "https://github.com/Uason-Chen/CTR-GCN/releases/download/v1.0/ctrgcn_ntu_120.pt"

In [ ]:
# 7. HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH
# Lưu output vào cả console và file text để bạn làm báo cáo
!python scripts/training/train_ctr_gcn.py | tee training_report.txt

In [ ]:
# 8. ĐÓNG GÓI VÀ SAO CHÉP MỌI KẾT QUẢ VỀ LẠI GOOGLE DRIVE
# a. Chép Model Weights (Cực kỳ quan trọng)
!cp weights/classification/ctrgcn_best.onnx /content/drive/MyDrive/Abnormal_Data/
!cp weights/classification/ctrgcn_best.pt /content/drive/MyDrive/Abnormal_Data/

# b. Chép Báo cáo đánh giá (Để copy dán vào Word/PDF)
!cp training_report.txt /content/drive/MyDrive/Abnormal_Data/

# c. Nén toàn bộ dữ liệu khung xương (.npy) đã xử lý để dành dùng lại sau này
!zip -r processed_data.zip data/processed/
!cp processed_data.zip /content/drive/MyDrive/Abnormal_Data/

print("TẤT CẢ KẾT QUẢ (Model, Báo Cáo, Data Khung Xương) ĐÃ ĐƯỢC LƯU VỀ GOOGLE DRIVE CỦA BẠN!")